# An Example Parquet Design Workflow for CCSPs

_The original intended audience of this notebook was Roman PITs creating file-based Parquet products (e.g., a file corresponds to a single light curve of a single source). Please consult with MAST to confirm that your file-based tabular  data is permitted to be formatted in Parquet._

_That said, much of the guidance contained herein will also apply to Roman PITs creating catalog Parquet products (e.g., a source photometry catalog spanning an entire survey). The only differences on the matters covered by this tutorial are that 1. true catalogs don't have to use both `table_meta_yaml` and VOParquet, only at least one of the two (though we recommend using both!) and 2. certain file-level metadata like RA, Dec, start time, end time, etc are not applicable to most true catalogs._

## Introduction

In this notebook, we'll show you how you can create a light curve file compliant with multiple guidelines: [MAST's PIT file design requirements](https://outerspace.stsci.edu/spaces/ISWG/pages/356669707/File+Design+for+PITs), the [ECSV-like table_meta_yaml approach](https://outerspace.stsci.edu/spaces/DraftMASTCONTRIB/pages/344588706/.File+Design+for+PITs+v1.0#id-.FileDesignforPITsv1.0-Column-levelMetadata) that Roman SOC Parquet files are taking, _and_ [VOParquet v1.0](https://www.ivoa.net/documents/Notes/VOParquet/), a convention for embedding richer and more standardized metadata in Parquet files.

First, we'll import everything that we need. Note that astropy >= v7.2.0 is required. Uncomment the following line to install all dependencies if needed, being careful to do so in a dedicated environment:

In [ ]:
# %pip install -r requirements_demo2.txt

In [ ]:
import numpy as np
import yaml
import pyarrow.parquet as pq
from astropy.io.votable import from_table, writeto, tree
from astropy.io import fits
from astropy.table import Table
from astropy import units as u
from astropy.time import Time

## Basic table_meta_yaml support

For the sake of this exercise, we'll start with a fake light curve in a draft FITS file adapted from a mission in development, `test-sparcs-jdref.fits`. For the purposes of this demonstration we'll be quite loose in our adaptation of this FITS file, using its metadata when it demonstrates a useful point, but using new metadata when it helps demonstrate a pattern that you are likely to use as a Roman PIT.

Let's open the FITS file up and retrieve its header metadata:

In [ ]:
hdul = fits.open('test-sparcs-jdref.fits')

hdul.info()

In [ ]:
header0 = hdul[0].header
header1 = hdul[1].header

Next, let's read the FITS BinTable into an Astropy `Table`:

In [ ]:
temp_cat = Table.read('test-sparcs-jdref.fits', unit_parse_strict = 'silent')

The approach that Astropy's FITS-parsing approach takes to masking missing values would overcomplicate things down the road, so let's fill in any masked values with NaNs:

In [ ]:
cat = temp_cat.filled(np.nan)

And let's also clean up a couple `deg_C` units that, because they contained underscores, are dicey in the [IVOA VOUnit syntax](https://ivoa.net/documents/VOUnits/). Because degrees Celsius are an unrecognized unit in the VO while 'd' is a recognized decimal prefix, we'll also need to put these units in single quotes, inside the strings, in order to prevent them from being parsed by VO applications as units of "deci-egCelsius".

In [ ]:
cat['TEMP_CCD'].unit = "'degCelsius'"
cat['TEMP_TEL'].unit = "'degCelsius'"
cat['TEMP_TEC'].unit = "'degCelsius'"
cat['TEMP_OPT'].unit = "'degCelsius'"
print(cat['TEMP_CCD'].unit)

Since we are leaving the FITS Standard and entering the realm of IVOA conventions, we'll want to assign IVOA Unified Content Descriptors (UCDs) to the important columns, when an appropriate UCD is available. These use an [internationally recognized standard](http://www.ivoa.net/Documents/latest/UCD.html) to tell software what each column contains, facilitating interoperability and data ingest, and thereby making your data more popular. Peruse the menu and syntax rules from the [IVOA's list of UCDs](https://ivoa.net/documents/UCD1+/), or ask MAST for advice.

In [ ]:
cat['TIME'].info.meta['ucd'] = 'time.epoch'
cat['CTR'].info.meta['ucd'] = 'phot.count'
cat['CTR_ERR'].info.meta['ucd'] = 'stat.error;phot.count'
cat['FLUX'].info.meta['ucd'] = 'phot.flux.density'
cat['FLUX_ERR'].info.meta['ucd'] = 'stat.error;phot.flux.density'
cat['EXPTIME'].info.meta['ucd'] = 'time.duration;obs.exposure'
cat['RA_AVG'].info.meta['ucd'] = 'pos.eq.ra'
cat['RA_SIG'].info.meta['ucd'] = 'stat.error;pos.eq.ra'
cat['DEC_AVG'].info.meta['ucd'] = 'pos.eq.dec'
cat['DEC_SIG'].info.meta['ucd'] = 'stat.error;pos.eq.dec'
cat['QUALITY'].info.meta['ucd'] = 'meta.code.qual'

We also need to give each column a description, which in this case we can lift from the FITS comments on the `TTYPEn` FITS keywords:

In [ ]:
# For each column in cat, assume the order hasn't changed and map it to a TTYPEn keyword, then get that keyword's comment
for n, colname in enumerate(cat.colnames, start=1):  # start=1 because python starts at 0 and FITS starts at TTYPE1
    keyword = f"TTYPE{n}"

    if keyword in header1:
        comment = header1.comments[keyword]  # get the FITS keyword record's comment
        if comment:
            comment = comment.replace('column name: ', '')  # cleanup to remove non sequiturs in this particular FITS file
            cat[colname].info.description = comment  # assign the FITS keyword's comment to this column in cat

For example, here's everything that we have added to the TIME column in the astropy `Table` `cat`:

In [ ]:
print(cat['TIME'].info)
print(cat['TIME'].meta)

Note that `cat` has also inherited a subselection of metadata from the FITS file, but we should deal with this more systematically, so we'll delete everything in `cat.meta` for now:

In [ ]:
cat.meta

In [ ]:
cat.meta = { }

Now we'll write `cat` to Parquet with a `table_meta_yaml` key. We haven't incorporated the VOParquet convention yet; we'll do that in the next section.

In [ ]:
cat.write('temp.parquet', format='parquet', overwrite=True)

Let's take a look at the contents of the `table_meta_yaml` key to see what column-level metadata we've populated. All of this is in a single string-valued key called `table_meta_yaml`, inside the key-value dictionary of the Parquet file's footer. Note that column metadata are rendered differently depending on whether a UCD has been assigned, but both syntaxes are fine.

In [ ]:
pqfile = pq.ParquetFile('temp.parquet')
keys_values = pqfile.metadata.metadata
yaml_str = keys_values[b'table_meta_yaml'].decode('utf-8')
print(yaml_str)

We've done a pretty tolerable job here already of providing column-level metadata that Astropy will understand, but we can do better by adding in metadata that _all IVOA-aware software_ will understand. We also would strongly benefit from some standardized way to specify the time scale, time reference position, and reference time of the `TIME` column, and to specify the coordinate system of the `RA_AVG` and `DEC_AVG` columns. All of these issues can be addressed by adding metadata content conformant to the [VOParquet](https://www.ivoa.net/documents/Notes/VOParquet/) convention.

## Adding VOParquet support

In brief, the idea behind VOParquet is that you take a single-tabled [VOTable](https://www.ivoa.net/documents/VOTable/)---an XML-flavored IVOA Standard for metadata-rich tables---and remove the data, leaving only the metadata. Then you pass the dataless XML VOTable into a single string-valued key in the Parquet footer's key-value dictionary.

To start, we'll convert the astropy `Table` into a `VOTableFile` object, using Astropy's `votable.from_table`:

In [ ]:
# Convert cat to a VOTableFile object
vocat = from_table(cat)

_(We can ignore these Astropy warnings here, which are overzealous; although `electron` is not recognized in VOUnit, the proper grammatical syntax for VOUnits is followed, and unrecognized units are permitted by VOUnit.)_

Now let's remove the data from the VOTable:

In [ ]:
for res in vocat.resources:
    for tab in res.tables:
        tab.array = np.zeros(0)  # zero-length array

We now have the skeleton of a table:

In [ ]:
display(vocat.get_first_table())

Next, let's add some metadata specifying how the timestamps should be interpreted, including:
- Time scale or time system (`timescale`), selected from the [IVOA Time Scales vocabulary](https://www.ivoa.net/rdf/timescale/). Common options include `UTC` or `TDB`.
- Time reference position (`refposition`), selected from the [IVOA Reference Positions vocabulary](https://www.ivoa.net/rdf/refposition/). Common options include `TOPOCENTER` or `BARYCENTER`.
- The reference time from which timestamps are offset (`timeorigin` in JD). In this case we'll use the standard `2400000.5` to indicate that timestamps are in boilerplate MJD. In the VOTable standard it is also permissible to specify the strings `MJD-origin` for vanilla MJD or `JD-origin` for vanilla JD.

To accomplish this, we'll create a TIMESYS element that can be referenced elsewhere in the VOTable as `ref.votable.coosys`.

In [ ]:
vocat.time_systems.append(tree.TimeSys(ID='ref.votable.timesys', refposition='TOPOCENTER', timescale='UTC', timeorigin=2400000.5))
print(vocat.time_systems)

_(Don't worry about warnings about VOTable versions at this stage; at the time of writing, Astropy will generate a v1.4 VOTable in the end.)_

Likewise, let's create a COOSYS element for the celestial coordinate system of spatial coordinates. We typically require that your coordinates be in `ICRS`, and the equinox should be `None` if the coordinate system is `ICRS`. The epoch we will also set to `None` in this case, because the sky coordinates for each timestamp correspond to the epoch of the timestamp, and VOTable doesn't have a rigorous way to indicate that.

In [ ]:
vocat.coordinate_systems.append(tree.CooSys(ID='ref.votable.coosys', epoch=None, equinox=None, system='ICRS'))
print(vocat.coordinate_systems)

Now we'll use referencing to indicate that the `TIME` column is in the time system we specified above (`ref.votable.timesys`), and that the RA and Dec columns are in the celestial coordinate system we specified above (`ref.votable.coosys`):

In [ ]:
vocat.get_field_by_id_or_name('TIME').ref = 'ref.votable.timesys'
vocat.get_field_by_id_or_name('RA_AVG').ref = 'ref.votable.coosys'
vocat.get_field_by_id_or_name('DEC_AVG').ref = 'ref.votable.coosys'

If you like, you can also populate other table-level metadata inside the VOTable, where we have the advantage of being able to specify units and reference coordinate systems (and even UCDs, if you like) in a rigorous way, and of being understand by IVOA-aware software. But MAST doesn't plan to be strict about this, as long as your VOTable has a `TIMESYS`, `COOSYS`, and column-level metadata. The following metadata will be duplicated in the top-level key-value dictionary of the Parquet footer regardless.

In [ ]:
# Numerical metadata or other complex non-string metadata can go in PARAM elements
# Single-valued string metadata can go in INFO elements

# Here are the metadata that often apply to all PIT parquet products, including both file-based products and catalogs

vocat.params.append(tree.Param(votable=vocat, name='start_time', value=header0['MJD-BEG'], unit=u.day, ref='ref.votable.timesys', datatype='double'))
vocat.params.append(tree.Param(votable=vocat, name='end_time', value=header0['MJD-END'], unit=u.day, ref='ref.votable.timesys', datatype='double'))
# vocat.params.append(tree.Param(votable=vocat, name='pixel_scale', value="something", unit=u.arcsec, datatype='float'))  # Characteristic pixel scale of the spatial axes, in arcsec per pixel. Omit if no spatial axes.
vocat.infos.append(tree.Info(name="ccsp.name", value="RGES"))
vocat.infos.append(tree.Info(name="ccsp.archive_lead", value="Baru Cormorant"))
vocat.infos.append(tree.Info(name="ccsp.investigator", value="Tain Hu"))
vocat.infos.append(tree.Info(name="ccsp.doi", value="10.17909/xxxx-xxxx"))
vocat.infos.append(tree.Info(name="file_version", value="v0.1"))
vocat.infos.append(tree.Info(name="file_date", value=str(Time(Time.now(), format='isot').value)))
vocat.infos.append(tree.Info(name="ccsp.data_release_id", value="DR0"))
vocat.infos.append(tree.Info(name="license", value="CC BY 4.0"))
vocat.infos.append(tree.Info(name="license_url", value="https://creativecommons.org/licenses/by/4.0/"))
vocat.infos.append(tree.Info(name="telescope", value="ROMAN"))
vocat.infos.append(tree.Info(name="instrument.name", value="WFI"))
# vocat.infos.append(tree.Info(name="instrument.detector", value="something"))  # Add back in if applicable
vocat.infos.append(tree.Info(name="instrument.optical_element", value="F062,F087"))  # Use a comma-separated list if necessary
vocat.infos.append(tree.Info(name="target_name", value="MWC 560"))
vocat.infos.append(tree.Info(name="wavelength.band", value="OPTICAL"))  # Use a comma-separated list if necessary
vocat.infos.append(tree.Info(name="ccsp.intent", value="SCIENCE"))
vocat.infos.append(tree.Info(name="target_keywords", value="Symbiotic binary stars,Stellar winds"))  # Select from https://astrothesaurus.org/concept-select/ , use a comma separated list if necessary
vocat.params.append(tree.Param(votable=vocat, name='target_keywords_id', value=[1674,1636], datatype='int', arraysize="*"))  # See ID in parentheses on concepts in https://astrothesaurus.org/concept-select/ , use a comma separated list if necessary
vocat.params.append(tree.Param(votable=vocat, name='wavelength.minimum', value=0.48, unit=u.micron,datatype='float'))
vocat.params.append(tree.Param(votable=vocat, name='wavelength.maximum', value=0.977, unit=u.micron,datatype='float'))
vocat.params.append(tree.Param(votable=vocat, name='simulated_flag', value=True, datatype='boolean'))  # Change to FALSE (or delete) for real data!
vocat.infos.append(tree.Info(name="source_observations_id", value="123456789, 987654321"))  # Use a comma separated list if necessary

# Here are the metadata that typically apply only to file-based products like light curves

vocat.params.append(tree.Param(votable=vocat, name='target_coordinates.ra', value=header0['RA_TARG'], unit=u.deg, ref='ref.votable.coosys', datatype='double'))
vocat.params.append(tree.Param(votable=vocat, name='target_coordinates.dec', value=header0['DEC_TARG'], unit=u.deg, ref='ref.votable.coosys', datatype='double'))
vocat.params.append(tree.Param(votable=vocat, name='exposure_time', value=header0['XPOSURE'], unit=u.s, datatype='double'))
# vocat.params.append(tree.Param(votable=vocat, name='effective_exposure_time', value=header0['XPOSURE'], unit=u.s, datatype='double'))  # Optional for time series, forbidden for coadds.
vocat.params.append(tree.Param(votable=vocat, name='cadence', value=90.0, unit=u.s, datatype='float'))
# vocat.infos.append(tree.Info(name="s_region", value="something"))  # OK to omit for point source products like 1D light curves and 1D spectra, but feel free to populate

When we're happy with our VOTable, we convert it to XML, then pass that XML into a string:

In [ ]:
# Write the VOTableFile object to XML
writeto(vocat, "vocat.xml")

# Read the XML as a string
with open("vocat.xml", "r", encoding="utf-8") as f:
    xml_string = f.read()

Let's take a look:

In [ ]:
print(xml_string)

## Putting it all together, and adding high-level metadata

Next, we pass that XML string into a `IVOA.VOTable-Parquet.content` key to be passed into the key-value dictionary in the Parquet footer. We also need a key indicating which version of the VOParquet convention we're using: at the time of writing, 1.0.

In [ ]:
# Create a dictionary to add keys to the Parquet footer
meta_dict = {}
meta_dict['IVOA.VOTable-Parquet.content'] = xml_string
meta_dict['IVOA.VOTable-Parquet.version'] = "1.0"

Finally, MAST requires that a set of high-level metadata be populated at the top level of the key-value dictionary in the Parquet footer. This may mirror some of the content you put in the VOTable key. Both keys and values must be strings in this case, unfortunately.

In [ ]:
# Here are the metadata that often apply to all PIT parquet products, including both file-based products and catalogs

meta_dict['ccsp.name'] = 'RGES'  # an example
meta_dict['ccsp.archive_lead'] = 'Baru Cormorant'
meta_dict['ccsp.investigator'] = 'Tain Hu'
meta_dict['ccsp.doi'] = '10.17909/xxxx-xxxx'
meta_dict['file_version'] = 'v0.1'
meta_dict['file_date'] = str(Time(Time.now(), format='isot').value)
meta_dict['ccsp.data_release_id'] = 'DR0'
meta_dict['license'] = 'CC BY 4.0'
meta_dict['license_url'] = 'https://creativecommons.org/licenses/by/4.0/'
meta_dict['telescope'] = 'ROMAN'
meta_dict['instrument.name'] = 'WFI'
# meta_dict['instrument.detector'] = 'something'  # Add back in if applicable
meta_dict['instrument.optical_element'] = 'F062,F087'  # Use a comma-separated list if necessary
meta_dict['target_name'] = 'MWC 560'
meta_dict['wavelength.band'] = 'OPTICAL' # Use a comma-separated list if necessary
meta_dict['wavelength.minimum'] = '0.48'
meta_dict['wavelength.maximum'] = '0.977'
meta_dict['ccsp.intent'] = 'SCIENCE'
meta_dict['start_time'] = str(header0['MJD-BEG'])
meta_dict['end_time'] = str(header0['MJD-END'])
meta_dict['simulated_flag'] = 'TRUE'  # Change to FALSE (or delete) for real data!
meta_dict['target_keywords'] = 'Symbiotic binary stars,Stellar winds'  # Select from https://astrothesaurus.org/concept-select/
meta_dict['target_keywords_id'] = '1674,1636'  # See ID in parentheses on concepts in https://astrothesaurus.org/concept-select/
meta_dict['source_observations_id'] = '123456789,987654321' # Use a comma-separated list if necessary
meta_dict['coordinate_reference_frame'] = 'ICRS'  # OK to omit in favor of COOSYS in VOTable, or duplicate
meta_dict['time_scale'] = 'UTC'  # OK to omit in favor of TIMESYS in VOTable, or duplicate
meta_dict['time_reference_position'] = 'TOPOCENTER'  # OK to omit in favor of TIMESYS in VOTable, or duplicate

# Here are the metadata that typically apply only to file-based products like light curves

meta_dict['exposure_time'] = str(header0['XPOSURE'])  # Summed across all time bins
# meta_dict['effective_exposure_time'] = 'something'  # Optional for time series, forbidden for coadds.
meta_dict['cadence'] = '90'  # Optional
meta_dict['target_coordinates.ra'] = str(header0['RA_TARG'])
meta_dict['target_coordinates.dec'] = str(header0['DEC_TARG'])
# meta_dict['s_region'] = 'something'  # OK to omit for point source products like 1D light curves and 1D spectra, but feel free to populate
# meta_dict['pixel_scale'] = 'something'  # Characteristic pixel scale of the spatial axes, in arcsec per pixel. Omit if no spatial axes.

To wrap it all up, we'll open back up the `temp.parquet` file, merge its metadata with `meta_dict`, and save to `vocat_final.parquet`:

In [ ]:
cat_pq = pq.read_table('temp.parquet')
existing = cat_pq.schema.metadata  # Get the metadata it already has
cat_pq = cat_pq.replace_schema_metadata({**existing, **meta_dict})  # Append the new VOParquet metadata keys
pq.write_table(cat_pq, 'vocat_final.parquet', compression=None)  # Save the file

Now we have a Parquet file that is compliant with multiple standards, including the international VOTable standard with VOParquet convention. The future scientists of the 24th century, trying to parse these ancient formats from before the cataclysm, thank you for your efforts!

_Coming soon: MAST will provide software that verifies whether your files are conformant to some of these conventions and requirements._